In [ ]:
import sys
import os
import math
from typing import NamedTuple

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold

import jax
import jax.numpy as jnp
import optax


sys.path.append(os.path.join(os.getcwd(), 'import'))

# noinspection PyUnresolvedReferences
from utils import read_utils
# noinspection PyUnresolvedReferences
from utils import plot_utils
# noinspection PyUnresolvedReferences
from utils import train_utils
# noinspection PyUnresolvedReferences
from utils import artificializer

# noinspection PyUnresolvedReferences
from models import bom_sep as bom
# noinspection PyUnresolvedReferences
from models import hgm_sep as hgm

# noinspection PyUnresolvedReferences
from models import bom as bom_gen
# noinspection PyUnresolvedReferences
from models import hgm as hgm_gen

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(artificializer)
importlib.reload(bom)
importlib.reload(hgm)
importlib.reload(bom_gen)
importlib.reload(hgm_gen)

# Functions

In [ ]:
class Lambdas(NamedTuple):
	reg_lambdas: tuple[float, float]

In [ ]:
def separate_losses(file_ids, columns, models_spn_res_pres):
	models_spn = [
		[
			pd.DataFrame(data=np.array(split[0])[None], columns=columns[i]) for split in models_spn_res_pres[i]
		] for i in range(len(file_ids))
	]
	models_pres = [
		[
			pd.DataFrame(data=np.array(split[1]), columns=columns[i]) for split in models_spn_res_pres[i]
		] for i in range(len(file_ids))
	]
	return models_spn, models_pres

In [ ]:
def assemble(boms_scores_per_neuron, hgms_scores_per_neuron):
	neurons_labels = [scores_per_neuron[0].columns for scores_per_neuron in hgms_scores_per_neuron]

	boms_scores_per_neuron = [np.array([split.values[0] for split in scores]) for scores in boms_scores_per_neuron]
	hgms_scores_per_neuron = [np.array([split.values[0] for split in scores]) for scores in hgms_scores_per_neuron]

	neuron_scores = dict()
	boms_scores_per_neuron_avg = [np.mean(scores, axis=0) for scores in boms_scores_per_neuron]
	hgms_scores_per_neuron_avg = [np.mean(scores, axis=0) for scores in hgms_scores_per_neuron]
	for label_list, score_bom_list, score_hgm_list in zip(neurons_labels, boms_scores_per_neuron_avg, hgms_scores_per_neuron_avg):
		for i in range(len(label_list)):
			label = label_list[i]
			if label != '' and label[-1] != '_':
				score_bom_new, score_hgm_new = np.array(score_bom_list[i]), np.array(score_hgm_list[i])
				if len(score_hgm_new.shape) == 0:
					score_bom_new = score_bom_new.reshape(1, 1)
					score_hgm_new = score_hgm_new.reshape(1, 1)
				if label not in neuron_scores.keys():
					neuron_scores[label] = (score_bom_new, score_hgm_new)
				else:
					score_bom_prev, score_hgm_prev = neuron_scores[label]
					neuron_scores[label] = (np.vstack([score_bom_prev, score_bom_new]), np.vstack([score_hgm_prev, score_hgm_new]))

	return neuron_scores

# read & prep

In [ ]:
default_hyperparameters = {
	'dimensions': 1,
	'w1_reg_lambda': 0.0,  #1E-3,
	'w2_reg_lambda': 0.0,  #1E-5,
}

bom_reg_lamdas = {
	# 'w1_reg_lambda': default_hyperparameters['w1_reg_lambda'],
	'w1_reg_lambda': 1E-2,
}
hgm_reg_lamdas = {
	# 'w1_reg_lambda': default_hyperparameters['w1_reg_lambda'],
	# 'w2_reg_lambda': default_hyperparameters['w2_reg_lambda'],
	'w1_reg_lambda': 3E-3,
	'w2_reg_lambda': 1E-7,
}


selection_n_scores = ['VB02', 'DB01', 'DD01', 'AVBR', 'AVBL',	# paper
					  'RMDDR', 'SMDDR', 'SMDDL',				# C
					  'AVAR', 'AVAL',							# M
					  'ALA', 'AWBR']							# O

selection_n_traces = ['AVAR', 'ALA',
					  'SMDDL', 'RMDDR', 'SMDDR', 'VB02', 'DB01',
					  'SIADR', 'DD01']
selection_bhl = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']



gcamp = True
read_ids_gcamp = list(range(0, 33+1))
# read_ids_gcamp = [0,   # minrevs
# 				  1,   # minswitch
# 				  7,   # maxswitch
# 				  11,  # median
# 				  12,  # maxrevs
# 				  17,  # mean
# 				  19,  # paper
# 				  21]  # clean
# read_ids_gcamp = [19]
plot_id_gcamp = 19


gfp = True
if gfp:
	read_ids_gfp = list(range(0, 7+1))
	# read_ids_gfp = [2, 4, 6]
	# read_ids_gfp = [2]
	plot_id_gfp = 2


artificial = True
if artificial:
	dims_generation = 2
	weights = (0.1, -0.25)

In [ ]:
# read

path = os.path.join(read_utils.get_data_path(), 'worm-2025-fieseler', 'exported_data_gcamp')
(file_ids_gcamp, filenames_gcamp,
 traces_gcamp,
 continuous_behaviors_gcamp,
 discrete_states_gcamp, eigenworms_gcamp,
 kymographs_gcamp) = read_utils.get_data(path, read_ids_gcamp)

# clear nan values
for i in range(len(file_ids_gcamp)):
	mask = eigenworms_gcamp[i].notna().all(axis=1)

	traces_gcamp[i] = traces_gcamp[i][mask]
	#continuous_behaviors_gcamp[i] = continuous_behaviors_gcamp[i][mask]
	eigenworms_gcamp[i] = eigenworms_gcamp[i][mask]
	discrete_states_gcamp[i] = discrete_states_gcamp[i][mask]

	#kymographs_gcamp[i] = kymographs_gcamp[i][mask]
	#filenames_gcamp[i] = filenames_gcamp[i][mask]

# remove unused discrete state labels
discrete_states_gcamp = [ds[selection_bhl] for ds in discrete_states_gcamp]

# naive ID unknown neurons
for i in range(len(file_ids_gcamp)):
	traces_gcamp[i].columns = pd.Index([
		f'{file_ids_gcamp[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces_gcamp[i].columns, traces_gcamp[i].columns == ''))
	])

modes_gcamp, _, traces_reconstructed_gcamp, _, _, _, _, _ = train_utils.get_manifolds_cca(traces_gcamp, discrete_states_gcamp, n_cv=default_hyperparameters['dimensions'], n_pc=0)
traces_residual_gcamp = [pd.DataFrame(data=trace_gcamp.values - traces_reconstructed_gcamp.values, columns=trace_gcamp.columns) for trace_gcamp, traces_reconstructed_gcamp in zip(traces_gcamp, traces_reconstructed_gcamp)]


if gfp:
	path_gfp = os.path.join(read_utils.get_data_path(), 'worm-2025-fieseler', 'exported_data_gfp')
	(file_ids_gfp, filenames_gfp,
	 traces_gfp,
	 continuous_behaviors_gfp,
	 discrete_states_gfp, eigenworms_gfp,
	 kymographs_gfp) = read_utils.get_data(path_gfp, read_ids_gfp)

	# clear nan values
	for i in range(len(file_ids_gfp)):
		mask = eigenworms_gfp[i].notna().all(axis=1)

		traces_gfp[i] = traces_gfp[i][mask]
		#continuous_behaviors_gfp[i] = continuous_behaviors_gfp[i][mask]
		eigenworms_gfp[i] = eigenworms_gfp[i][mask]
		discrete_states_gfp[i] = discrete_states_gfp[i][mask]

		#kymographs_gfp[i] = kymographs_gfp[i][mask]
		#filenames_gfp[i] = filenames_gfp[i][mask]

	# remove unused discrete state labels
	discrete_states_gfp = [ds[selection_bhl] for ds in discrete_states_gfp]

	# naive ID unknown neurons
	for i in range(len(file_ids_gfp)):
		traces_gfp[i].columns = pd.Index([
			f'{file_ids_gfp[i]:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces_gfp[i].columns, traces_gfp[i].columns == ''))
		])

	modes_gfp, _, traces_reconstructed_gfp, _, _, _, _, _ = train_utils.get_manifolds_cca(traces_gfp, discrete_states_gfp, n_cv=default_hyperparameters['dimensions'], n_pc=0)
	traces_residual_gfp = [pd.DataFrame(data=trace_gfp.values - traces_reconstructed_gfp.values, columns=trace_gfp.columns) for trace_gfp, traces_reconstructed_gfp in zip(traces_gfp, traces_reconstructed_gfp)]


# generate
if artificial:
	file_ids_art = list()
	filenames_art = list()

	discrete_states_art = list()
	eigenworms_art = list()

	modes_art = list()
	traces_reconstructed_art = list()
	traces_residual_art = list()
	traces_art = list()

	for i in range(len(file_ids_gcamp)):
		(filename_art,
		 neural_modes_art_,
		 traces_reconstructed_art_,
		 traces_residual_art_,
		 traces_art_,
		 params_art_,
		 params_cca_art_) = artificializer.artificialize(weights, hgm_gen,
														 filenames_gcamp[i],
														 traces_gcamp[i].columns,
														 eigenworms_gcamp[i],
														 discrete_states_gcamp[i],
														 n_dimensions=dims_generation)
		filenames_art.append(filename_art)

		modes_art.append(neural_modes_art_)
		traces_reconstructed_art.append(traces_reconstructed_art_)
		traces_residual_art.append(traces_residual_art_)
		traces_art.append(traces_art_)

		file_ids_art.append(file_ids_gcamp[i])

		discrete_states_art.append(discrete_states_gcamp[i].copy())
		eigenworms_art.append(eigenworms_gcamp[i].copy())

In [ ]:
selection_n_traces_gcamp = [label for label in selection_n_traces if label in traces_gcamp[file_ids_gcamp.index(plot_id_gcamp)].columns]
if gfp:
	selection_n_traces_gfp = [label for label in selection_n_traces if label in traces_gfp[file_ids_gfp.index(plot_id_gfp)].columns]

# read models

In [ ]:
prefix = 'allfiles-ll0-d2'

In [ ]:
if gcamp:
	bom_load_name_gcamp = os.path.join(prefix, f'sep_bom_gcamp_w1-{bom_reg_lamdas['w1_reg_lambda']:.0E}')
	hgm_load_name_gcamp = os.path.join(prefix, f'sep_hgm_gcamp_w1-{hgm_reg_lamdas['w1_reg_lambda']:.0E}_w2-{hgm_reg_lamdas['w2_reg_lambda']:.0E}')

if gfp:
	bom_load_name_gfp = os.path.join(prefix, f'sep_bom_gfp_w1-{bom_reg_lamdas['w1_reg_lambda']:.0E}')
	hgm_load_name_gfp = os.path.join(prefix, f'sep_hgm_gfp_w1-{hgm_reg_lamdas['w1_reg_lambda']:.0E}_w2-{hgm_reg_lamdas['w2_reg_lambda']:.0E}')

if artificial:
	bom_load_name_art = os.path.join(prefix, f'sep_bom_art_w1-{bom_reg_lamdas['w1_reg_lambda']:.0E}')
	hgm_load_name_art = os.path.join(prefix, f'sep_hgm_art_w1-{hgm_reg_lamdas['w1_reg_lambda']:.0E}_w2-{hgm_reg_lamdas['w2_reg_lambda']:.0E}')

In [ ]:
if gcamp:
	boms_params_gcamp = train_utils.load_params(bom_load_name_gcamp, filenames_gcamp)

	boms_losses_gcamp_all = train_utils.load_losses(bom_load_name_gcamp, filenames_gcamp)
	boms_losses_train_gcamp = [losses[0] for losses in boms_losses_gcamp_all]
	boms_mpe_losses_train_gcamp = [losses[1] for losses in boms_losses_gcamp_all]
	boms_losses_gcamp = [losses[2] for losses in boms_losses_gcamp_all]
	boms_mpe_losses_gcamp = [losses[3] for losses in boms_losses_gcamp_all]

	hgms_params_gcamp = train_utils.load_params(hgm_load_name_gcamp, filenames_gcamp)

	hgms_losses_gcamp_all = train_utils.load_losses(hgm_load_name_gcamp, filenames_gcamp)
	hgms_losses_train_gcamp = [losses[0] for losses in hgms_losses_gcamp_all]
	hgms_mpe_losses_train_gcamp = [losses[1] for losses in hgms_losses_gcamp_all]
	hgms_losses_gcamp = [losses[2] for losses in hgms_losses_gcamp_all]
	hgms_mpe_losses_gcamp = [losses[3] for losses in hgms_losses_gcamp_all]


if gfp:
	boms_params_gfp = train_utils.load_params(bom_load_name_gfp, filenames_gfp)

	boms_losses_gfp_all = train_utils.load_losses(bom_load_name_gfp, filenames_gfp)
	boms_losses_train_gfp = [losses[0] for losses in boms_losses_gfp_all]
	boms_mpe_losses_train_gfp = [losses[1] for losses in boms_losses_gfp_all]
	boms_losses_gfp = [losses[2] for losses in boms_losses_gfp_all]
	boms_mpe_losses_gfp = [losses[3] for losses in boms_losses_gfp_all]

	hgms_params_gfp = train_utils.load_params(hgm_load_name_gfp, filenames_gfp)

	hgms_losses_gfp_all = train_utils.load_losses(hgm_load_name_gfp, filenames_gfp)
	hgms_losses_train_gfp = [losses[0] for losses in hgms_losses_gfp_all]
	hgms_mpe_losses_train_gfp = [losses[1] for losses in hgms_losses_gfp_all]
	hgms_losses_gfp = [losses[2] for losses in hgms_losses_gfp_all]
	hgms_mpe_losses_gfp = [losses[3] for losses in hgms_losses_gfp_all]


if artificial:
	boms_params_art = train_utils.load_params(bom_load_name_art, filenames_art)

	boms_losses_art_all = train_utils.load_losses(bom_load_name_art, filenames_art)
	boms_losses_train_art = [losses[0] for losses in boms_losses_art_all]
	boms_mpe_losses_train_art = [losses[1] for losses in boms_losses_art_all]
	boms_losses_art = [losses[2] for losses in boms_losses_art_all]
	boms_mpe_losses_art = [losses[3] for losses in boms_losses_art_all]

	hgms_params_art = train_utils.load_params(hgm_load_name_art, filenames_art)

	hgms_losses_art_all = train_utils.load_losses(hgm_load_name_art, filenames_art)
	hgms_losses_train_art = [losses[0] for losses in hgms_losses_art_all]
	hgms_mpe_losses_train_art = [losses[1] for losses in hgms_losses_art_all]
	hgms_losses_art = [losses[2] for losses in hgms_losses_art_all]
	hgms_mpe_losses_art = [losses[3] for losses in hgms_losses_art_all]

# artificial

### losses

In [ ]:
if artificial:
	plot_utils.plot_losses_splits('art hgm vs bom: train',
								  filenames_art,
								  hgms_losses_train_art, boms_losses_train_art,
								  ylimits=(0, 0.3))

In [ ]:
if artificial:
	plot_utils.plot_losses_splits('art hgm vs bom: test',
								  filenames_art,
								  hgms_losses_art, boms_losses_art,
								  ylimits=(0, 0.3))

### predict & score

In [ ]:
if artificial:
	splits_art = [list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces_art[i].shape[0]), discrete_states_art[i][['REV']].values)) for i in range(len(file_ids_art))]

In [ ]:
if artificial:
	boms_spn_pres_art = [[
		bom.score_fn(params, traces_residual_art[i].values[eval], eigenworms_art[i].values[eval]) for params, (train, eval) in zip(boms_params_art[i], splits_art[i])
	] for i in range(len(file_ids_art))]

	hgms_spn_pres_art = [[
		hgm.score_fn(params, traces_residual_art[i].values[eval], eigenworms_art[i].values[eval], modes_art[i].values[eval]) for params, (train, eval) in zip(hgms_params_art[i], splits_art[i])
	] for i in range(len(file_ids_art))]

	(boms_spn_art,
	 boms_pres_art) = separate_losses(file_ids_art, [trace.columns for trace in traces_art], boms_spn_pres_art)

	(hgms_spn_art,
	 hgms_pres_art) = separate_losses(file_ids_art, [trace.columns for trace in traces_art], hgms_spn_pres_art)

### performance

In [ ]:
if artificial:
	neuron_scores_art = assemble(boms_spn_art, hgms_spn_art)

In [ ]:
if artificial:
	plot_utils.plot_performance_plot(neuron_scores_art, None,
									 title='scores for all neurons', limits=(-0.55, 1.1, -0.55, 1.1))

In [ ]:
if artificial:
	plot_utils.plot_performance_plot({key: neuron_scores_art[key] for key in neuron_scores_art.keys() if not key[0].isdigit()}, None,
									 title='scores for ID\'d neurons', limits=(-0.55, 1.1, -0.55, 1.1))

In [ ]:
# if artificial:
# 	plot_utils.plot_performance_plot({key: neuron_scores_art[key] for key in selection_n_scores if key in neuron_scores_art.keys()}, None,
# 								 	title='scores for handpicked neurons', limits=(-0.55, 1.1, -0.55, 1.1))

### traces

In [ ]:
if artificial:
	boms_spn_pres_art = [[
		bom.score_fn(params, traces_residual_art[i].values, eigenworms_art[i].values) for params in boms_params_art[i]
	] for i in range(len(file_ids_art))]

	hgms_spn_pres_art = [[
		hgm.score_fn(params, traces_residual_art[i].values, eigenworms_art[i].values, modes_art[i].values) for params in hgms_params_art[i]
	] for i in range(len(file_ids_art))]

	(boms_spn_art_full,
	 boms_pres_art_full) = separate_losses(file_ids_art, [trace.columns for trace in traces_art], boms_spn_pres_art)

	(hgms_spn_art_full,
	 hgms_pres_art_full) = separate_losses(file_ids_art, [trace.columns for trace in traces_art], hgms_spn_pres_art)

In [ ]:
if artificial:
	for i in range(5):
		plot_utils.plot_traces_comparisons_sep(
			f'{filenames_art[file_ids_art.index(plot_id_gcamp)]} split {i} art hgm & bom predictions residuals',
			discrete_states_art[file_ids_art.index(plot_id_gcamp)][['REV']],
			traces_residual_art[file_ids_art.index(plot_id_gcamp)][selection_n_traces_gcamp],
			hgms_pres_art_full[file_ids_art.index(plot_id_gcamp)][i][selection_n_traces_gcamp],
			boms_pres_art_full[file_ids_art.index(plot_id_gcamp)][i][selection_n_traces_gcamp],
			plot_first=True, plots_per_row=1, ylimits=None,
			save_name=f'SEP_ART_PRES_S{i}')  #None

 # gcamp

### losses

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits('gcamp hgm vs bom: train',
								  filenames_gcamp,
								  hgms_losses_train_gcamp, boms_losses_train_gcamp,
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits('gcamp hgm vs bom: mpe train',
								  filenames_gcamp,
								  hgms_mpe_losses_train_gcamp, boms_mpe_losses_train_gcamp,
								  ylimits=(0.05, 0.09))

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits('gcamp hgm vs bom: test',
								  filenames_gcamp,
								  hgms_losses_gcamp, boms_losses_gcamp,
								  ylimits=None)

In [ ]:
if gcamp:
	plot_utils.plot_losses_splits('gcamp hgm vs bom: mpe test',
								  filenames_gcamp,
								  hgms_mpe_losses_gcamp, boms_mpe_losses_gcamp,
								  ylimits=(0.04, 0.10))

### predict & score

In [ ]:
if gcamp:
	splits_gcamp = [list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces_gcamp[i].shape[0]), discrete_states_gcamp[i][['REV']].values)) for i in range(len(file_ids_gcamp))]

In [ ]:
if gcamp:
	boms_spn_pres_gcamp = [[
		bom.score_fn(params, traces_residual_gcamp[i].values[eval], eigenworms_gcamp[i].values[eval]) for params, (train, eval) in zip(boms_params_gcamp[i], splits_gcamp[i])
	] for i in range(len(file_ids_gcamp))]

	hgms_spn_pres_gcamp = [[
		hgm.score_fn(params, traces_residual_gcamp[i].values[eval], eigenworms_gcamp[i].values[eval], modes_gcamp[i].values[eval]) for params, (train, eval) in zip(hgms_params_gcamp[i], splits_gcamp[i])
	] for i in range(len(file_ids_gcamp))]

	(boms_spn_gcamp,
	 boms_pres_gcamp) = separate_losses(file_ids_gcamp, [trace.columns for trace in traces_gcamp], boms_spn_pres_gcamp)

	(hgms_spn_gcamp,
	 hgms_pres_gcamp) = separate_losses(file_ids_gcamp, [trace.columns for trace in traces_gcamp], hgms_spn_pres_gcamp)

### performance

In [ ]:
if gcamp:
	neuron_scores_gcamp = assemble(boms_spn_gcamp, hgms_spn_gcamp)

In [ ]:
if gcamp:
	plot_utils.plot_performance_plot(neuron_scores_gcamp, None,
									 title='scores for all neurons', limits=(-0.55, 0.55, -0.55, 0.55))

In [ ]:
if gcamp:
	plot_utils.plot_performance_plot({key: neuron_scores_gcamp[key] for key in neuron_scores_gcamp.keys() if not key[0].isdigit()}, None,
								 	title='scores for ID\'d neurons', limits=(-0.55, 0.55, -0.55, 0.55))

In [ ]:
if gcamp:
	plot_utils.plot_performance_plot({key: neuron_scores_gcamp[key] for key in selection_n_scores if key in neuron_scores_gcamp.keys()}, None,
								 	title='scores for handpicked neurons', limits=(-0.55, 0.55, -0.55, 0.55))

### traces

In [ ]:
if gcamp:
	boms_spn_pres_gcamp = [[
		bom.score_fn(params, traces_residual_gcamp[i].values, eigenworms_gcamp[i].values) for params in boms_params_gcamp[i]
	] for i in range(len(file_ids_gcamp))]

	hgms_spn_pres_gcamp = [[
		hgm.score_fn(params, traces_residual_gcamp[i].values, eigenworms_gcamp[i].values, modes_gcamp[i].values) for params in hgms_params_gcamp[i]
	] for i in range(len(file_ids_gcamp))]

	(boms_spn_gcamp_full,
	 boms_pres_gcamp_full) = separate_losses(file_ids_gcamp, [trace.columns for trace in traces_gcamp], boms_spn_pres_gcamp)

	(hgms_spn_gcamp_full,
	 hgms_pres_gcamp_full) = separate_losses(file_ids_gcamp, [trace.columns for trace in traces_gcamp], hgms_spn_pres_gcamp)

In [ ]:
if gcamp:
	for i in range(5):
		plot_utils.plot_traces_comparisons_sep(
			f'{filenames_gcamp[file_ids_gcamp.index(plot_id_gcamp)]} split {i} gcamp hgm & bom predictions residuals',
			discrete_states_gcamp[file_ids_gcamp.index(plot_id_gcamp)][['REV']],
			traces_residual_gcamp[file_ids_gcamp.index(plot_id_gcamp)][selection_n_traces_gcamp],
			hgms_pres_gcamp_full[file_ids_gcamp.index(plot_id_gcamp)][i][selection_n_traces_gcamp],
			boms_pres_gcamp_full[file_ids_gcamp.index(plot_id_gcamp)][i][selection_n_traces_gcamp],
			plot_first=True, plots_per_row=1, ylimits=None,
			save_name=f'SEP_GCAMP_PRES_S{i}')  #None

# gfp

### predict & score

In [ ]:
if gfp:
	splits_gfp = [list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces_gfp[i].shape[0]), discrete_states_gfp[i][['REV']].values)) for i in range(len(file_ids_gfp))]

In [ ]:
if gfp:
	boms_spn_pres_gfp = [[
		bom.score_fn(params, traces_residual_gfp[i].values[eval], eigenworms_gfp[i].values[eval]) for params, (train, eval) in zip(boms_params_gfp[i], splits_gfp[i])
	] for i in range(len(file_ids_gfp))]

	hgms_spn_pres_gfp = [[
		hgm.score_fn(params, traces_residual_gfp[i].values[eval], eigenworms_gfp[i].values[eval], modes_gfp[i].values[eval]) for params, (train, eval) in zip(hgms_params_gfp[i], splits_gfp[i])
	] for i in range(len(file_ids_gfp))]

	(boms_spn_gfp,
	 boms_pres_gfp) = separate_losses(file_ids_gfp, [trace.columns for trace in traces_gfp], boms_spn_pres_gfp)

	(hgms_spn_gfp,
	 hgms_pres_gfp) = separate_losses(file_ids_gfp, [trace.columns for trace in traces_gfp], hgms_spn_pres_gfp)

### performance

In [ ]:
if gfp:
	neuron_scores_gfp = assemble(boms_spn_gfp, hgms_spn_gfp)

In [ ]:
if gcamp and gfp:
	plot_utils.plot_performance_plot(neuron_scores_gcamp, neuron_scores_gfp,
									 title='scores for all neurons (+ GFP)', limits=(-0.55, 0.55, -0.55, 0.55))

In [ ]:
if gcamp and gfp:
	high_score_hgm_gfp = np.quantile(np.concatenate([pair[1] for pair in neuron_scores_gfp.values()]), 0.95)

	selection_n_hgmssig = list()
	for key in neuron_scores_gcamp.keys():
		scores_bom, scores_hgm = neuron_scores_gcamp[key]
		if len(scores_hgm.shape) == 0:
			scores_bom = scores_bom.reshape(1, 1)
			scores_hgm = scores_hgm.reshape(1, 1)
		y = np.median(scores_hgm)
		if y > high_score_hgm_gfp:
			selection_n_hgmssig.append(key)

In [ ]:
# if gcamp and gfp:
# 	neuron_imprsig_gcamp = {key: np.mean(neuron_scores_gcamp[key][1] - neuron_scores_gcamp[key][0]) for key in selection_n_hgmssig if key in neuron_scores_gcamp.keys()}
# 	selection_n_imprsig = [key for _, key in reversed(sorted(zip(neuron_imprsig_gcamp.values(), neuron_imprsig_gcamp.keys())))]

In [ ]:
if gcamp and gfp:
	plot_utils.plot_performance_plot({key: neuron_scores_gcamp[key] for key in selection_n_hgmssig if key in neuron_scores_gcamp.keys()}, None,
									 title='scores for significant neurons', limits=(-0.55, 0.55, -0.55, 0.55))

In [ ]:
if gcamp and gfp:
	plot_utils.plot_performance_plot({key: neuron_scores_gcamp[key] for key in selection_n_hgmssig if key in neuron_scores_gcamp.keys() and not key[0].isdigit()}, None,
									 title='scores for significant ID\'d neurons', limits=(-0.55, 0.55, -0.55, 0.55))

In [ ]:
if gcamp and gfp:
	plot_utils.plot_improvement_plot_y({key: neuron_scores_gcamp[key] for key in selection_n_hgmssig if key in neuron_scores_gcamp.keys()}, None,
									   title='score improvements for significant neurons', limits=(-0.55, 0.55, -0.22, 0.44))

In [ ]:
if gcamp and gfp:
	plot_utils.plot_improvement_plot_y({key: neuron_scores_gcamp[key] for key in selection_n_hgmssig if key in neuron_scores_gcamp.keys() and not key[0].isdigit()}, None,
									   title='score improvements for significant ID\'d neurons', limits=(-0.55, 0.55, -0.15, 0.30))

In [ ]:
if gcamp and gfp:
	print('\n'.join([f'NEURON: {key},\tIMPR: {np.median(val[1] - val[0]):.4f}' for key, val in reversed(sorted(neuron_scores_gcamp.items(), key=lambda x: np.median(x[1][1] - x[1][0]))) if key in selection_n_hgmssig and not key[0].isdigit()]))